# 05 — qwen1b SFT adapters, TQ1_0

Select a **Colab GPU** kernel in VS Code. Read `04_quantized_setup.ipynb` for the laptop receiver, tunnel, and audible watchdog. Use only the pinned frozen base/adapters. Inference sends the same complete, unmodified teacher-row prompt with the frozen chat template, greedy decoding, seed 42, repetition penalty 1 and 1,024-token maximum. Every row is an independent llama-server completion with prompt caching disabled; prompt identity and tokenization are checked. An empty or malformed model response is still a completed response. Context-unsupported rows retain an explicit status. This condition does not use CAM-LDS importance-matrix calibration.

**Inference process revision (2 November 2026).** Earlier saved 04 responses used one llama-server slot and synchronously acknowledged each row through the laptop receiver. The revised process uses model-specific concurrent slots for independent one-row requests, atomic Colab records, background receiver uploads in batches of up to five, backpressure at 25 outstanding rows, and laptop CSV snapshots every 25 mutations. GGUF construction, rendered prompts, frozen HF token IDs, per-row context allowance, and generation settings are unchanged. The old and new processes can coexist within the same resumable stream; the per-row parallel_slots field distinguishes new records. Before the revision, this 05 notebook had 0/285 response rows saved for each of its four adapter streams, according to results/quantized record files. This refers to inference results, not preparation of adapter GGUF artifacts.

## 1. Restore code and data from laptop

Start the extended receiver, tunnel and watchdog using notebook 04 setup. Paste the HTTPS URL and receiver token. The receiver sends frozen code, manifests, prompts and tokenizer files; it does not send API/HF secrets in the bundle.

**Recovery / safe rerun:** New runtime or tunnel: rerun 1, then 2, 3, and the interrupted stage. The same URL/token may be reused across different models.

**Cost:** No paid API requests.



In [2]:
from pathlib import Path
from getpass import getpass
import json, io, os, sys, urllib.request, zipfile
ROOT = Path('/content/CAM_LDS_STREAMLINED')
ROOT.mkdir(parents=True, exist_ok=True)
receiver_url = input('Local receiver HTTPS tunnel URL: ').strip().rstrip('/')
receiver_token = getpass('Local receiver token: ').strip()
from urllib.parse import urlparse
if urlparse(receiver_url).scheme != 'https':
    raise ValueError('Enter the https:// URL printed by cloudflared')
request = urllib.request.Request(receiver_url+'/bundle', headers={'Authorization':'Bearer '+receiver_token})
with urllib.request.urlopen(request, timeout=120) as response:
    bundle = response.read()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.infolist():
        target = (ROOT/member.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid bundle path')
    archive.extractall(ROOT)
(ROOT/'runtime').mkdir(exist_ok=True)
connection = ROOT/'runtime/connection.json'
connection.write_text(json.dumps({'url':receiver_url,'token':receiver_token}), encoding='utf-8')
os.chmod(connection, 0o600)
del receiver_token, bundle
sys.path.insert(0, str(ROOT))
print('Bootstrap restored. No API/HF key is stored in notebook output.')

Bootstrap restored. No API/HF key is stored in notebook output.


## 2. Install pinned worker requirements

**Installations and built toolchains are reused within the same Colab runtime. A new runtime reinstalls Python packages but restores the locally saved, pinned CUDA build only when its GPU architecture and CUDA toolkit match.** The frozen base model is downloaded directly from Hugging Face and cached within that runtime. Final quantized artifacts are saved on the laptop.

**Recovery / safe rerun:** After runtime reset rerun 1 then 2; ordinary cell reruns reuse installed packages.

**Cost:** No paid API requests.



In [3]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-colab.txt')],check=True)
print('Worker dependencies ready')

Worker dependencies ready


## 3. Verify receiver, watchdog, configuration and GPU

The same authenticated receiver serves all distinct model notebooks. Check GPU identity and available memory before construction.

**Recovery / safe rerun:** After connection errors rerun 1 and 3. After a new runtime rerun 1–3. Resolve duplicate live worker leases before retrying.

**Cost:** No paid API requests.



In [4]:
from streamlined.common import read_json, verify_experiment, CONDITIONS
from streamlined.transport import Mirror
from continuation.notebook import worker
from continuation.protocol import condition_key
import subprocess
MODEL='qwen1b'
REPRESENTATION='tq1_0'
connection=read_json(ROOT/'runtime/connection.json')
mirror=Mirror(connection['url'],connection['token'])
health=mirror.call('/health')
if not health.get('watchdog_alive'):raise RuntimeError('Start the laptop audible watchdog')
print(health, verify_experiment(ROOT)['config_hash'])
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'],text=True))

{'protocol': 'camlds-streamlined-v1', 'watchdog_alive': True} 49e13df7571913d2c39067dfd7dcde93353110b2f4c7523bc8a18de2459b4b73
NVIDIA L4, 23034 MiB, 580.82.07



## 4. Prepare the pinned GGUF artifacts

Merge each complete HF adapter into its frozen BF16 base, convert to GGUF, then construct TQ1_0. The laptop receives each final GGUF artifact and its checksum before the notebook advances. **This stage can spend substantial GPU/CPU time and local disk; completed artifacts and toolchains are reused on rerun within a live runtime.** BitNet SFT export uses a compatibility gate and stops with a clear error if this pinned native path cannot convert the merged checkpoint.

**Recovery / safe rerun:** Rerun 1–3 after a runtime change, then 4. A fully uploaded final artifact is reused; failed conversions leave no completed artifact manifest.

**Cost:** GPU/CPU construction and potentially multi-GB upload; no paid Haiku calls.



In [4]:
for adapter in CONDITIONS:
    worker(ROOT,MODEL,REPRESENTATION,adapter,action='prepare')

Started quantized/qwen1b/sft_technique/tq1_0/prepare; PID=24326; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_technique_tq1_0_prepare.log
Reused laptop CUDA toolchain for architecture 89
I0000 00:00:1793750547.149745   24326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793750547.221266   24326 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793750549.776546   24326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orde

## 5. Five-row inference pilot

Run five frozen IDs per stream through the final server and save them as real results. The full run skips these IDs. The GGUF tokenizer is compared to the frozen HF tokenizer before generation. Frozen HF token IDs define the experimental input. If the GGUF pre-tokenizer segments identical text differently, the backend must detokenize the frozen IDs back to the exact rendered prompt before generation; the difference is retained in a tokenizer audit. The pilot uses the same per-request context and model-specific parallel-slot policy as the full run. Each row remains a separate request and context. Inspect the server log for actual CUDA offload and token/s before the long run.

**Recovery / safe rerun:** Rerun 1–3 after a new runtime, then 5. Saved pilot IDs are not repeated.

**Cost:** GPU inference for up to five rows per stream.



In [5]:
for adapter in CONDITIONS:
    worker(ROOT,MODEL,REPRESENTATION,adapter,action='infer',pilot=5)

Started quantized/qwen1b/sft_technique/tq1_0/infer; PID=52188; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_technique_tq1_0_infer_pilot5.log
quantized/qwen1b/sft_technique/tq1_0: ID 2 saved; 1/5
quantized/qwen1b/sft_technique/tq1_0: ID 14 saved; 5/5
Started quantized/qwen1b/sft_explanation/tq1_0/infer; PID=54340; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_explanation_tq1_0_infer_pilot5.log
quantized/qwen1b/sft_explanation/tq1_0: ID 12 saved; 1/5
quantized/qwen1b/sft_explanation/tq1_0: ID 2 saved; 2/5
quantized/qwen1b/sft_explanation/tq1_0: ID 14 saved; 5/5
Started quantized/qwen1b/sft_joint/tq1_0/infer; PID=56532; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_joint_tq1_0_infer_pilot5.log
quantized/qwen1b/sft_joint/tq1_0: ID 5 saved; 1/5
quantized/qwen1b/sft_joint/tq1_0: ID 2 saved; 2/5
quantized/qwen1b/sft_joint/tq1_0: ID 14 saved; 5/5
Started quantized/qwen1b/sft_full/tq1_0/infer; PID=58653; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwe

## 6. Complete independent inference

All four adapters are run on the frozen validation and test IDs only. Independent rows are submitted concurrently using a conservative model-specific slot count; completion order may differ, but every result is keyed to its frozen prompt ID. Each response is first written atomically as a JSON record on Colab. A background uploader sends laptop checkpoints in batches of five, applies backpressure at 25 queued rows, and fully flushes at completion. The laptop CSV snapshot is refreshed every 25 saved rows and at normal completion. The same prompt IDs are skipped on resume; errors are retryable. GPU, latency, token and I/O metrics are saved as telemetry.

**Recovery / safe rerun:** After interruption rerun 1–3 and 6. Rerun 4 only if the GGUF artifact was never completed; section 5 is optional on resume. The per-row JSON records, including rows after the latest CSV snapshot, are authoritative.

**Cost:** GPU inference for remaining rows; no paid Haiku calls.



In [25]:
for adapter in CONDITIONS:
    worker(ROOT,MODEL,REPRESENTATION,adapter,action='infer')

Started quantized/qwen1b/sft_technique/tq1_0/infer; PID=304084; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_technique_tq1_0_infer.log
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/CAM_LDS_STREAMLINED/continuation/infer.py", line 375, in <module>
    if __name__=='__main__':main()
                            ~~~~^^
  File "/content/CAM_LDS_STREAMLINED/continuation/infer.py", line 352, in main
    with FileLock(str(root/'runtime'/f'{args.model}.quantized.lock'),timeout=0):
         ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 951, in __enter__
    self.acquire()
    ~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 1059, in acquire
    self._poll_until_acquired(
    ~~~~~~~~~~~~~~~~~~~~~~~~~^
        blocking=blocking,
        ^^^^

RuntimeError: quantized/qwen1b/sft_technique/tq1_0/infer exited 1; see /content/CAM_LDS_STREAMLINED/telemetry/logs/qwen1b_sft_technique_tq1_0_infer.log

## 7. Verify coverage and snapshot

Read current laptop-backed record counts. Check that each required stream is terminal before launching the Haiku judge.

**Recovery / safe rerun:** Rerun 1–3 then 7 after a runtime change. Missing IDs are handled by rerunning 6.

**Cost:** No paid API requests.



In [30]:
from continuation.store import SyncStore
from continuation.protocol import selected_rows, stream
from collections import Counter
for adapter in (CONDITIONS if 'CONDITIONS' in globals() else __import__('streamlined.common',fromlist=['CONDITIONS']).CONDITIONS) if True else (None,):
    name=stream(MODEL,REPRESENTATION,adapter)
    with SyncStore(ROOT,name,mirror) as store:
        store.snapshot()
        expected=len(selected_rows(ROOT,REPRESENTATION,adapter))
        states=Counter(row['status'] for row in store.rows.values())
        print(name,dict(states), 'expected:',expected,'saved:',len(store.rows))

quantized/qwen1b/sft_technique/tq1_0 {'completed': 285} expected: 285 saved: 285
quantized/qwen1b/sft_explanation/tq1_0 {'completed': 285} expected: 285 saved: 285
quantized/qwen1b/sft_joint/tq1_0 {'completed': 285} expected: 285 saved: 285
quantized/qwen1b/sft_full/tq1_0 {'completed': 285} expected: 285 saved: 285
